### General Requirements 

In [ ]:
%pip install nlp

In [4]:
%pip install beautifulsoup4
%pip install lxml
%pip install spacy
%pip install ipywidgets
%pip install transformers

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
     |████████████████████████████████| 139 kB 2.6 MB/s eta 0:00:01
     |████████████████████████████████| 214 kB 17.6 MB/s eta 0:00:01
     |████████████████████████████████| 2.3 MB 54.3 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [45]:
%pip install beautifulsoup4
%pip install lxml

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### Open the dataset

In [140]:
from bs4 import BeautifulSoup


# Reading the data inside the xml
# file to a variable under the name
# data
with open('deid_surrogate_train_all_version2.xml', 'r') as f:
    data = f.read()

# Passing the stored data inside
# the beautifulsoup parser, storing
# the returned object
Bs_data = BeautifulSoup(data, "xml")

# Using find() to extract attributes
# of the first instance of the tag
b_type = Bs_data.find_all('PHI', {'TYPE':'HOSPITAL'})

print(b_type)

[<PHI TYPE="HOSPITAL">FIH</PHI>, <PHI TYPE="HOSPITAL">Sephsandpot Center</PHI>, <PHI TYPE="HOSPITAL">Valtawnprinceel Community Memorial Hospital</PHI>, <PHI TYPE="HOSPITAL">Valtawnprinceel
            Community Memorial Hospital</PHI>, <PHI TYPE="HOSPITAL">Em Nysonken Medical Center</PHI>, <PHI TYPE="HOSPITAL">OLH</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">Staviewordna University Of Medical Center</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">6U-489</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">Hoseocon Medical Center</PHI>, <PHI TYPE="HOSPITAL">Heaonboburg Linpack Grant Medical Center</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">Liccam Community Medical
            Center</PHI>, <PHI TYPE="HOSPITAL">Liccam Community Medical Center</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">1D-419</PHI>, <PHI TYPE="HOSPITAL"

### Prepare as text

In [141]:
from spacy import displacy
import re

xml_text = Bs_data.get_text()
record_test = Bs_data.find('RECORD')

### Remove IDs if needed

In [142]:
def remove_ids(soup):
    for item in soup.find_all(attrs={"TYPE": "ID"}):
        item.string = "*****"
    return soup

In [143]:
record_test = remove_ids(record_test) # Remove IDs from the XML data
record_text = record_test.get_text()
record_str = str(record_test)

In [144]:
nlp.max_length = len(record_str) + 100  # add a bit of a buffer
doc_xml = nlp(record_str)

#displacy.serve(doc_xml, style="ent")

In [145]:
# Define the entities to mask
entities_to_mask = ["PERSON", "EMAIL", "GPE", "DATE", "LOC", "FAC"]
pattern_date = re.compile("[0-9]{2}\/[0-9]{2}\/[0-9]{2,4}")

entities_recognized = set()

# Function to mask entities
def mask_entities(doc, entities_to_mask):
    masked_text = doc.text
    for ent in doc.ents:
        if ent.label_ in entities_to_mask:
            entities_recognized.add(ent)
            masked_text = masked_text.replace(ent.text, "*****")
        if pattern_date.match(ent.text):
            masked_text = masked_text.replace(ent.text, "DATE")
    
    print(entities_recognized)
    return masked_text


In [146]:
# Mask the entities
masked_text = mask_entities(doc_xml, entities_to_mask)

# Print the masked text
print(masked_text)

masked_xml = nlp(masked_text)
#displacy.serve(masked_xml, style="ent")

{Cultures, 2+, 42, Haldol, Isordil, ST, Morphine, Lasix, ST, CO2 57, Micronase, Unasyn, 32, 84, Discharge Summary Unsigned, 79-year-old, Gelfoam, M.D., Versed, ST, 30 years, Lasix, 38, 6, the 16th, prothrombin, Cimetidine 300, 15 years, 150/60, Clindamycin, Mucomist, 28, Unasyn}
<RECORD ID="*****40">
<TEXT>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="HOSPITAL">FIH</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="DATE">11/19</PHI>/1994
            12:00:00 AM ***** DIS Report Status : Unsigned ADMISSION DATE : <PHI TYPE="DATE">11/19</PHI>/94 DISCHARGE DATE : <PHI TYPE="DATE">11/*****</PHI>/94
            ADMISSION DIAGNOSIS : Aspiration pneumonia , esophageal laceration . HI*****ORY OF PRESENT
            ILLNESS : Mr. <PHI TYPE="PATIENT">Blind</PHI> is a ***** white white male with a
            history of diabetes mellitus , inferior myocardial infarction , who underwent open
            repair of his increased diverticulum <PHI TYPE="DATE">N

Retransform to xml file

In [111]:
import xml.etree.ElementTree as ET

# Parse the XML string
root = ET.fromstring(masked_text)

# Create an ElementTree object
tree = ET.ElementTree(root)

# Write the ElementTree object to an XML file
tree.write("output.xml", encoding="utf-8", xml_declaration=True)

# Print the content of the XML file
with open("output.xml", "r") as f:
    print(f.read())

<?xml version='1.0' encoding='utf-8'?>
<RECORD ID="*****40">
<TEXT>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="HOSPITAL">FIH</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="DATE">11/19</PHI>/1994
            12:00:00 AM ***** DIS Report Status : Unsigned ADMISSION DATE : <PHI TYPE="DATE">11/19</PHI>/94 DISCHARGE DATE : <PHI TYPE="DATE">11/*****</PHI>/94
            ADMISSION DIAGNOSIS : Aspiration pneumonia , esophageal laceration . HI*****ORY OF PRESENT
            ILLNESS : Mr. <PHI TYPE="PATIENT">Blind</PHI> is a ***** white white male with a
            history of diabetes mellitus , inferior myocardial infarction , who underwent open
            repair of his increased diverticulum <PHI TYPE="DATE">November 13th</PHI> at <PHI TYPE="HOSPITAL">Sephsandpot Center</PHI> . The patient developed hematemesis <PHI TYPE="DATE">November 15th</PHI> and was intubated for respiratory distress . He was
            transferred to the <PHI TYPE="HOSPIT

# Bio NER

### Load the specialized model

In [3]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("token-classification", model="alvaroalon2/biobert_diseases_ner")

/home/cbrice/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


# Finetuning BERT

### Requirements

In [135]:
%pip install scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [134]:
%pip install transformers
%pip install torch


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


### Fine tune

#### Load dataset

In [136]:
import xml.etree.ElementTree as ET
import re
from transformers import BertTokenizer

# Function to extract and clean text from XML file
def extract_text_from_xml(xml_file):
    # Parse the XML file
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Function to extract all text from XML tags
    def get_text_from_element(element):
        if element.text:
            return element.text.strip()
        return ""
    
    # Extract text from each element
    all_text = []
    for elem in root.iter():
        text = get_text_from_element(elem)
        if text:
            all_text.append(text)
    
    # Join the extracted text
    return " ".join(all_text)

# Function to clean the extracted text (optional)
def clean_text(text):
    # Remove unwanted characters like extra spaces or special characters
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove non-alphanumeric characters
    return text.strip()

# Function to tokenize the text for BERT fine-tuning
def tokenize_text(text, tokenizer):
    # Tokenize the cleaned text using BERT tokenizer
    tokens = tokenizer(text, padding=True, truncation=True, return_tensors='pt')
    return tokens

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Example usage
xml_file = 'output.xml'  # Path to your XML file
extracted_text = extract_text_from_xml(xml_file)
cleaned_text = clean_text(extracted_text)
tokens = tokenize_text(cleaned_text, tokenizer)

# Example: print the tokenized inputs for inspection
print(tokens)


{'input_ids': tensor([[  101, 10882,  2232, 11118,  2683, 11118,  2683,  2340,  6397,  2281,
          6122, 19802,  7898,  5685, 11008,  2415,  2281,  6286, 11748,  2696,
          7962, 18098,  2378,  3401,  2884,  2451,  3986,  2902,  1015,  2705,
          1997,  2281,  7398, 13816, 11748,  2696,  7962, 18098,  2378,  3401,
          2884,  2451,  3986,  2902,  7861,  6396,  3385,  7520,  2966,  2415,
         13928, 13386, 10965, 19330,  2232, 10965, 10965,  1016,  2705,  1016,
          2705,  1016,  2705,  1016,  2705, 17089,  1062,  2666,  1049,  4224,
          1053,  2361,  1038, 18938,  7287,  2629,  7287,  2620,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

#### Tokenisation

#### Fine-tuning

In [138]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForMaskedLM, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split

# Step 1: Prepare the dataset

# Let's assume `cleaned_text` is a list of text samples
texts = [cleaned_text]  # This should be a list of text data, e.g., ["text1", "text2", ...]

# Split the data into training and validation sets
train_texts, val_texts = train_test_split(texts, test_size=0.1, train_size=0.9)

# Step 2: Create a custom Dataset for BERT

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        # Tokenize the text and add special tokens
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        input_ids = encoding['input_ids'].squeeze(0)  # Remove batch dimension
        attention_mask = encoding['attention_mask'].squeeze(0)
        return {'input_ids': input_ids, 'attention_mask': attention_mask}

# Initialize the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Create DataLoader objects
train_dataset = TextDataset(train_texts, tokenizer)
val_dataset = TextDataset(val_texts, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=8)

# Step 3: Load a pre-trained BERT model

# You can use BERT for masked language modeling (BERT's default task)
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

# Step 4: Set up the optimizer, loss function, and scheduler

# Optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Total number of training steps (based on batch size and number of epochs)
total_steps = len(train_dataloader) * 3  # Let's assume 3 epochs

# Learning rate scheduler
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# Step 5: Train the model

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training loop
for epoch in range(3):  # Loop over the dataset multiple times
    model.train()
    for batch in train_dataloader:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask, labels=input_ids)
        
        # Compute loss (Masked Language Modeling loss)
        loss = outputs.loss
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        # Print loss (optional)
        print(f"Epoch {epoch+1}, Loss: {loss.item()}")

    # Validate the model (optional)
    model.eval()
    total_eval_loss = 0
    for batch in val_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask, labels=input_ids)
        
        total_eval_loss += outputs.loss.item()

    print(f"Validation Loss after epoch {epoch+1}: {total_eval_loss / len(val_dataloader)}")

# Step 6: Save the fine-tuned model

model.save_pretrained('fine_tuned_bert')
tokenizer.save_pretrained('fine_tuned_bert')


ValueError: With n_samples=1, test_size=0.1 and train_size=0.9, the resulting train set will be empty. Adjust any of the aforementioned parameters.

#### Evaluate the model